In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense

# 1. Load IMDB Review Dataset (CSV)
data = pd.read_csv("IMDB Dataset.csv")
print(data.head())
print("Dataset Shape:", data.shape)

# Convert sentiment text to numbers: positive = 1, negative = 0
data["label"] = data["sentiment"].map({"positive": 1, "negative": 0})

X = data["review"].values
y = data["label"].values

# Split into training and testing data
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training Dataset Size:", len(X_train_text))
print("Testing Dataset Size:", len(X_test_text))

# 2. Tokenize text and apply padding
vocab_size = 10000
max_length = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

X_train = tokenizer.texts_to_sequences(X_train_text)
X_test = tokenizer.texts_to_sequences(X_test_text)

X_train = pad_sequences(X_train, maxlen=max_length, padding="post", truncating="post")
X_test = pad_sequences(X_test, maxlen=max_length, padding="post", truncating="post")

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

# 3. Build LSTM Model
model = Sequential([
    Input(shape=(max_length,)),
    Embedding(input_dim=vocab_size, output_dim=64, mask_zero=True),
    LSTM(64),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

# 4. Compile Model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("\nLSTM Model Summary:")
model.summary()

# 5. Train Model
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# 6. Evaluate Model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print("\nMODEL EVALUATION")
print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")

# 7. Plot Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("LSTM Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

# 8. Display Actual and Predicted Sentiment
predictions = model.predict(X_test[:10], verbose=0)
predicted_labels = (predictions >= 0.5).astype(int).flatten()

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)

    actual = "Positive" if y_test[i] == 1 else "Negative"
    predicted = "Positive" if predicted_labels[i] == 1 else "Negative"

    plt.text(0.5, 0.6, "Actual:\n" + actual, ha="center")
    plt.text(0.5, 0.3, "Predicted:\n" + predicted, ha="center")
    plt.axis("off")

plt.suptitle("Actual vs Predicted Sentiment")
plt.tight_layout()
plt.show()